# Application Project - SingularIT

Imports:

In [2]:
import pandas as pd
import numpy as np
import json
import glob
import os

In [3]:
folder_path = 'game_results/results' #data folder
file_pattern = os.path.join(folder_path, '*.json') #specifying json data format
json_files = glob.glob(file_pattern) 

all_dataframes = []

for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)

        df_temp = pd.json_normalize(
            data['runs'], 
            record_path=['events'], 
            meta=['run_id', 'player_1', 'player_2']
        )

        df_temp['source_file'] = os.path.basename(file)
            
        all_dataframes.append(df_temp)
            
timestamped_data = pd.concat(all_dataframes, ignore_index=True)

In [4]:
timestamped_data

,timestamp,player,type,bar,side,successful,run_id,player_1,player_2,source_file
0,2.985530,Diana,contact,Middle1,Middle,NaN,0,Diana,Hans,Diana-Hans.json
1,3.749828,Diana,contact,Attack1,Right,NaN,0,Diana,Hans,Diana-Hans.json
2,6.102422,Hans,contact,Attack2,Middle,NaN,0,Diana,Hans,Diana-Hans.json
3,10.453081,Hans,shot,NaN,NaN,True,0,Diana,Hans,Diana-Hans.json
4,20.291921,Diana,contact,Middle1,Middle,NaN,0,Diana,Hans,Diana-Hans.json
...,...,...,...,...,...,...,...,...,...,...
29332,910.737441,Tanja,contact,Defense2,Middle,NaN,4,Simon,Tanja,Simon-Tanja.json
29333,911.646976,Tanja,shot,NaN,NaN,False,4,Simon,Tanja,Simon-Tanja.json
29334,911.837511,Simon,contact,Goal1,Middle,NaN,4,Simon,Tanja,Simon-Tanja.json
29335,913.151401,Simon,contact,Defense1,Left,NaN,4,Simon,Tanja,Simon-Tanja.json


In [5]:
timestamped_data.shape

(29337, 10)

In [6]:
def create_results_table(df):
    is_shot = (df['type'] == 'shot')
    is_goal = (df['successful'] == True)
    
    df['is_p1_goal'] = is_shot & is_goal & (df['player'] == df['player_1'])
    df['is_p2_goal'] = is_shot & is_goal & (df['player'] == df['player_2'])

    results = df.groupby(['source_file', 'run_id']).agg({
        'player_1': 'first',
        'player_2': 'first',
        'is_p1_goal': 'sum',
        'is_p2_goal': 'sum'
    }).reset_index()

    results = results.rename(columns={
        'is_p1_goal': 'points_player_1',
        'is_p2_goal': 'points_player_2'
    })

    results['winner'] = np.where(
        results['points_player_2'] > results['points_player_1'], results['player_2'], results['player_1'])
    
    results['loser'] = np.where(
        results['points_player_2'] > results['points_player_1'], results['player_1'], results['player_2'])

    df.drop(columns=['is_p1_goal', 'is_p2_goal'], inplace=True)

    return results

results_table = create_results_table(timestamped_data)

results_table

,source_file,run_id,player_1,player_2,points_player_1,points_player_2,winner,loser
0,Diana-Hans.json,0,Diana,Hans,10,5,Diana,Hans
1,Diana-Hans.json,1,Diana,Hans,10,5,Diana,Hans
2,Diana-Hans.json,2,Diana,Hans,10,5,Diana,Hans
3,Diana-Hans.json,3,Diana,Hans,10,4,Diana,Hans
4,Diana-Hans.json,4,Diana,Hans,10,5,Diana,Hans
...,...,...,...,...,...,...,...,...
70,Simon-Tanja.json,0,Simon,Tanja,10,8,Simon,Tanja
71,Simon-Tanja.json,1,Simon,Tanja,9,10,Tanja,Simon
72,Simon-Tanja.json,2,Simon,Tanja,5,10,Tanja,Simon
73,Simon-Tanja.json,3,Simon,Tanja,10,6,Simon,Tanja


In [7]:
results_table['winner'].value_counts().reset_index()

,winner,count
0,Diana,22
1,Simon,17
2,Olga,15
3,Magnus,10
4,Tanja,8
5,Hans,3


In [8]:
def create_win_matrix(df):
    players = sorted(pd.unique(df[['player_1','player_2']].to_numpy().ravel()))

    win_matrix = (
        pd.crosstab(df.winner, df.loser)
        .reindex(index=players, columns=players, fill_value=0)
        .astype(float)
    )
    win_matrix.where(~np.eye(len(players), dtype=bool), np.nan, inplace=True)
    return win_matrix

win_matrix = create_win_matrix(results_table)
win_matrix

loser,Diana,Hans,Magnus,Olga,Simon,Tanja
winner,,,,,,
Diana,NaN,5.0,4.0,5.0,3.0,5.0
Hans,0.0,NaN,0.0,1.0,0.0,2.0
Magnus,1.0,5.0,NaN,1.0,1.0,2.0
Olga,0.0,4.0,4.0,NaN,2.0,5.0
Simon,2.0,5.0,4.0,3.0,NaN,3.0
Tanja,0.0,3.0,3.0,0.0,2.0,NaN


### Rutvik and Cosima

In [ ]:
timestamped_data['is_goal'] = (timestamped_data['type'] == 'shot') & (timestamped_data['successful'] == True)

timestamped_data['round'] = 0

for (source_file, run_id), group in timestamped_data.groupby(['source_file', 'run_id']):
    current_round = 1
    rounds = []

    for is_goal in group['is_goal'].shift(fill_value=False):
        if is_goal:
            current_round += 1
        rounds.append(current_round)

    timestamped_data.loc[group.index, 'round'] = rounds

def aggregate_rounds(group):
    p1 = group['player_1'].iloc[0]
    p2 = group['player_2'].iloc[0]

    goal_rows = group.loc[group['is_goal']]
    round_winner = goal_rows['player'].iloc[0] if not goal_rows.empty else None

    start_ts = group['timestamp'].iloc[0]
    end_ts = group['timestamp'].iloc[-1]
    duration = group['timestamp'].iloc[-1] - group['timestamp'].iloc[0]

    contacts_player1 = len(group[(group['player'] == p1) & (group['type'] == 'contact')])
    contacts_player2 = len(group[(group['player'] == p2) & (group['type'] == 'contact')])

    shots_player1 = len(group[(group['player'] == p1) & (group['type'] == 'shot')])
    shots_player2 = len(group[(group['player'] == p2) & (group['type'] == 'shot')])

    return pd.Series({
        'player_1': p1,
        'player_2': p2,
        'round_winner': round_winner,
        'start': start_ts,
        'end': end_ts,
        'duration': duration,
        'contacts_player1': contacts_player1,
        'contacts_player2': contacts_player2,
        'shots_player1': shots_player1,
        'shots_player2': shots_player2,
    })

round_data = (
    timestamped_data
    .groupby(['source_file', 'run_id', 'round'], as_index=False)
    .apply(aggregate_rounds)
    .reset_index(drop=True)
)


ValueError: Can only compare identically-labeled Series objects

In [36]:
round_data.head(20)

,source_file,run_id,round,player_1,player_2,round_winner,start,end,duration,contacts_player1,contacts_player2,shots_player1,shots_player2
0,Diana-Hans.json,0,1,Diana,Hans,Hans,2.985530,10.453081,7.467551,2,1,0,1
1,Diana-Hans.json,0,2,Diana,Hans,Diana,20.291921,51.365341,31.073420,5,5,3,2
2,Diana-Hans.json,0,3,Diana,Hans,Diana,56.560030,150.898025,94.337994,15,17,5,7
3,Diana-Hans.json,0,4,Diana,Hans,Diana,164.168779,178.463446,14.294667,3,2,3,1
4,Diana-Hans.json,0,5,Diana,Hans,Hans,191.487753,208.643816,17.156064,4,5,3,2
5,Diana-Hans.json,0,6,Diana,Hans,Diana,217.202873,271.257861,54.054988,6,10,4,2
6,Diana-Hans.json,0,7,Diana,Hans,Diana,281.351738,300.194387,18.842649,7,1,3,0
7,Diana-Hans.json,0,8,Diana,Hans,Diana,307.495760,344.197300,36.701540,13,8,2,3
8,Diana-Hans.json,0,9,Diana,Hans,Diana,351.500147,426.960668,75.460522,11,15,4,5
9,Diana-Hans.json,0,10,Diana,Hans,Hans,432.869073,463.302999,30.433926,3,5,1,4
